<hr style="border: 8px solid#0B0B0B;" />
<br>
<div align="center">
    <img src="images/radar_plots_many.png" align="center">
</div>

<br>
<br>

<div align="left" style="font-size:32px; font-family:FreeMono; font-weight: 100; font-stretch:ultra-condensed; line-height:1.2; color:#2A2C2B">
    <strong>BRIEF</strong> OVERVIEW OF <br> RADAR <strong>PLOTS</strong>
</div>

<br>

**Learning Goal(s):** Learn how to build radial (radar) plots using Matplotlib - from a simple polar subplot to a fully custom projection with polygon framing.

**Target:** Developers and data scientists who need to visualize multi-dimensional comparisons on a single chart.

**Prerequisite Knowledge:** (1) [Matplotlib](https://matplotlib.org/) and (2) [Pandas](https://pandas.pydata.org/)

**License:** MIT - use freely with attribution.

<hr style="border: 4px solid#0B0B0B;" />

## **Radial Plot Basic**

Multiple Axes (plots) on same figure. Code is taken from https://python-graph-gallery.com/391-radar-chart-with-several-individuals/

In [1]:
import os
import matplotlib.pyplot as plt
import pandas as pd
from math import pi

os.makedirs("images", exist_ok=True)

df = pd.DataFrame({
    'group': ['A', 'B', 'C', 'D'],
    'var1': [38, 1.5, 30, 4],
    'var2': [29, 10, 9, 34],
    'var3': [8, 39, 23, 24],
    'var4': [7, 31, 33, 14],
    'var5': [28, 15, 32, 14]
})

categories = list(df)[1:]
N = len(categories)

angles = [n / float(N) * 2 * pi for n in range(N)]
angles += angles[:1]

ax = plt.subplot(111, polar=True)
ax.set_theta_offset(pi / 2)
ax.set_theta_direction(-1)

plt.xticks(angles[:-1], categories)

ax.set_rlabel_position(0)
plt.yticks([10, 20, 30], ["10", "20", "30"], color="grey", size=7)
plt.ylim(0, 40)

values = df.loc[0].drop("group").values.flatten().tolist()
values += values[:1]
ax.plot(angles, values, linewidth=1, linestyle="solid", label="group A")
ax.fill(angles, values, "b", alpha=0.1)

values = df.loc[1].drop("group").values.flatten().tolist()
values += values[:1]
ax.plot(angles, values, linewidth=1, linestyle="solid", label="group B")
ax.fill(angles, values, "r", alpha=0.1)

plt.legend(loc="upper right", bbox_to_anchor=(0.1, 0.1))
plt.savefig("images/radar_plot_basic.png", bbox_inches="tight", facecolor="white")
plt.show()


<hr style="border: 4px solid#0B0B0B;" />

## **Radial Plot Advanced**

Multiple Axes (plots) on same figure, but only display portion of axes containted within polygon. 

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.path import Path
from matplotlib.spines import Spine
from matplotlib.projections.polar import PolarAxes
from matplotlib.projections import register_projection

os.makedirs("images", exist_ok=True)


def radar_factory(num_vars, frame="circle"):
    """Create a radar chart with `num_vars` axes.

    Parameters
    ----------
    num_vars : int
        Number of variables for the radar chart.
    frame : {"circle" | "polygon"}
        Shape of frame surrounding axes.
    """
    theta = np.linspace(0, 2 * np.pi, num_vars, endpoint=False)
    theta += np.pi / 2

    def draw_poly_patch(self):
        verts = unit_poly_verts(theta)
        return plt.Polygon(verts, closed=True, edgecolor="#4E5E6E")

    def draw_circle_patch(self):
        return plt.Circle((0.5, 0.5), 0.5)

    patch_dict = {"polygon": draw_poly_patch, "circle": draw_circle_patch}
    if frame not in patch_dict:
        raise ValueError("unknown value for `frame`: %s" % frame)

    class RadarAxes(PolarAxes):
        name = "radar"
        RESOLUTION = 1
        draw_patch = patch_dict[frame]

        def fill(self, *args, **kwargs):
            closed = kwargs.pop("closed", True)
            return super(RadarAxes, self).fill(closed=closed, *args, **kwargs)

        def plot(self, *args, **kwargs):
            lines = super(RadarAxes, self).plot(*args, **kwargs)
            for line in lines:
                self._close_line(line)

        def _close_line(self, line):
            x, y = line.get_data()
            if x[0] != x[-1]:
                x = np.concatenate((x, [x[0]]))
                y = np.concatenate((y, [y[0]]))
                line.set_data(x, y)

        def set_varlabels(self, labels):
            self.set_thetagrids(np.degrees(theta), labels)

        def _gen_axes_patch(self):
            return self.draw_patch()

        def _gen_axes_spines(self):
            if frame == "circle":
                return PolarAxes._gen_axes_spines(self)
            spine_type = "circle"
            verts = unit_poly_verts(theta)
            verts.append(verts[0])
            path = Path(verts)
            spine = Spine(self, spine_type, path)
            spine.set_transform(self.transAxes)
            return {"polar": spine}

    register_projection(RadarAxes)
    return theta


def unit_poly_verts(theta):
    """Return vertices of polygon for subplot axes.

    The polygon is circumscribed by a unit circle centered at (0.5, 0.5).
    """
    x0, y0, r = 0.5, 0.5, 0.5
    return [(r * np.cos(t) + x0, r * np.sin(t) + y0) for t in theta]


# --- Sample data: two teams rated across six dimensions (scale 0-10) ---
spoke_labels = ["Collaboration", "Innovation", "Resilience",
                "Communication", "Adaptability", "Leadership"]

team_a_scores = [7.2, 8.5, 6.1, 7.8, 9.0, 6.5]
team_b_scores = [6.0, 7.0, 6.5, 7.0, 7.5, 8.0]

N = len(spoke_labels)
theta = radar_factory(N, frame="polygon")

fig = plt.figure(figsize=(8, 8), dpi=120)
ax = fig.add_subplot(111, projection="radar")
plt.rgrids([2, 4, 6, 8, 10])

color_a = "#275170"
color_b = "#F77F03"

ax.plot(theta, team_a_scores, color=color_a, linewidth=1.5)
ax.fill(theta, team_a_scores, facecolor=color_a, alpha=0.25)

ax.plot(theta, team_b_scores, color=color_b, linewidth=1.5)
ax.fill(theta, team_b_scores, facecolor=color_b, alpha=0.25)

ax.set_varlabels(spoke_labels)
ax.set_ylim([0, 10])

patch_a = mpatches.Patch(color=color_a, alpha=0.85, label="Team A")
patch_b = mpatches.Patch(color=color_b, alpha=0.85, label="Team B")
lgd = plt.legend(handles=[patch_a, patch_b], loc=(0.85, 1.05), frameon=True)

tit = plt.suptitle("Performance Radar - Team Comparison", fontsize=14,
                   fontweight="bold", y=1.02)

plt.tight_layout()
plt.savefig("images/radar_plot_advanced.png",
            bbox_extra_artists=(lgd, tit), bbox_inches="tight",
            facecolor="white", edgecolor="none")
plt.show()
